In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [3]:
df = spark.read.csv('olist_orders_dataset.csv', header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------

In [4]:
from pyspark.sql.functions import col, to_timestamp, coalesce

# Converter as colunas de data que estão como String para datetime
df = df.withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss")) \
       .withColumn("order_approved_at", to_timestamp(col("order_approved_at"), "yyyy-MM-dd HH:mm:ss")) \
       .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss")) \
       .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss")) \
       .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss"))

# Filtrar apenar order_status == ‘delivered’
df = df.filter(col("order_status") == "delivered")

In [5]:
# Exibir o schema e os dados transformados
df.printSchema()
df.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------

In [6]:
# Remover linhas vazias de ‘order_delivered_customer_date’
df = df.na.drop(subset=["order_delivered_customer_date"])

# Se a coluna ‘order_approved_at’ estiver vazia, preencher com o valor da ‘order_purchase_timestamp’
df = df.withColumn("order_approved_at", coalesce(col("order_approved_at"), col("order_purchase_timestamp")))

# Remover a coluna order_status
df = df.drop("order_status")

# Exibir o schema e os dados transformados
df.printSchema()
df.show(5)


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------+--------------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|     2017-10-02 10:56:33|2017-10

In [7]:
# Salvar o DataFrame agregado em CSV usando Pandas (evita dependencia do Hadoop no Windows)
import csv
import os
import pandas as pd
from datetime import datetime

orders_final_df = df.toPandas()
output_dir = os.getcwd()
output_path = os.path.join(output_dir, f"orders_final_{datetime.now():%Y%m%d_%H%M%S}.csv")
orders_final_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\orders_final_20260331_230351.csv
